# Notebook 12 — Ultimate Adversarial Training Study

**Four training regimes, one notebook. 25 new runs total.**

| Group | Technique | Backbone | Runs |
|---|---|---|---|
| A | FGM ε ∈ {0.3, 0.5*, 0.8, 1.0} | BanglaBERT | 13 |
| B | FreeLB K=3 | BanglaBERT | 4 |
| C | FGM + AWP | BanglaBERT | 4 |
| D | FGM ε=0.5 cross-backbone | MuRIL | 4 |

\* ε=0.5 binary already done in nb06 — loaded, not re-trained.

**Theoretical hierarchy:** FGM < FreeLB < FGM+AWP (in principle).  
This notebook empirically ranks all three on Bengali sarcasm for the first time.

In [37]:
# ── Cell 1: Imports, environment detection, paths ──────────────────────────
import os, gc, json, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    set_seed
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings('ignore')
set_seed(42)

# ── Environment detection ──────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    ENV       = 'kaggle'
    REPO      = '/kaggle/working/Sarcasm_detection'
    CACHE_DIR = '/kaggle/tmp/hf_cache'
    CKPT_DIR  = '/kaggle/tmp/checkpoints'
else:
    ENV       = 'local'
    REPO      = os.path.abspath(os.path.join(os.getcwd(), '..'))
    CACHE_DIR = None
    CKPT_DIR  = os.path.join(REPO, '03_models', 'checkpoints', 'tmp_nb12')

os.makedirs(CKPT_DIR, exist_ok=True)
if CACHE_DIR:
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME']            = CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
    os.environ['HF_DATASETS_CACHE']  = CACHE_DIR

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_FP16 = (DEVICE == 'cuda')

# ── Output dirs ───────────────────────────────────────────────────────────
DATA_SPLITS = os.path.join(REPO, '01_data', 'interim', 'splits')
OUT_DIR     = os.path.join(REPO, '04_outputs')
TABLES_DIR  = os.path.join(OUT_DIR, 'tables')
PREDS_DIR   = os.path.join(OUT_DIR, 'predictions')
REPORTS_DIR = os.path.join(OUT_DIR, 'reports')
LOGS_DIR    = os.path.join(OUT_DIR, 'run_logs')
for d in [TABLES_DIR, PREDS_DIR, REPORTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

OUTPUT_CSV = os.path.join(TABLES_DIR, '12_adv_training_results.csv')

print(f'ENV      = {ENV}')
print(f'DEVICE   = {DEVICE}  (FP16={USE_FP16})')
print(f'REPO     = {REPO}')

ENV      = local
DEVICE   = cpu  (FP16=False)
REPO     = /Users/sefayet/Desktop/Github/Sarcasm_detection


In [38]:
# ── Cell 2: Run matrix ─────────────────────────────────────────────────────
BANGLABERT = 'csebuetnlp/banglabert'
MURIL      = 'google/muril-base-cased'

# Columns: run_id | technique | backbone_short | backbone_hf | param | dataset | task
# param = epsilon for fgm, K for freelb, epsilon for fgm_awp
RUN_MATRIX = [
    # ── Group A: BanglaBERT FGM — epsilon study ───────────────────────────
    ('bb_fgm_e03_banglasarc_binary',     'fgm', 'banglabert', BANGLABERT, 0.3, 'banglasarc',  'binary'),
    ('bb_fgm_e03_banglasarc3_binary',    'fgm', 'banglabert', BANGLABERT, 0.3, 'banglasarc3', 'binary'),
    ('bb_fgm_e03_banglasarc3_ternary',   'fgm', 'banglabert', BANGLABERT, 0.3, 'banglasarc3', 'ternary'),
    ('bb_fgm_e03_ben_sarc_binary',       'fgm', 'banglabert', BANGLABERT, 0.3, 'ben_sarc',    'binary'),
    ('bb_fgm_e05_banglasarc3_ternary',   'fgm', 'banglabert', BANGLABERT, 0.5, 'banglasarc3', 'ternary'),
    ('bb_fgm_e08_banglasarc_binary',     'fgm', 'banglabert', BANGLABERT, 0.8, 'banglasarc',  'binary'),
    ('bb_fgm_e08_banglasarc3_binary',    'fgm', 'banglabert', BANGLABERT, 0.8, 'banglasarc3', 'binary'),
    ('bb_fgm_e08_banglasarc3_ternary',   'fgm', 'banglabert', BANGLABERT, 0.8, 'banglasarc3', 'ternary'),
    ('bb_fgm_e08_ben_sarc_binary',       'fgm', 'banglabert', BANGLABERT, 0.8, 'ben_sarc',    'binary'),
    ('bb_fgm_e10_banglasarc_binary',     'fgm', 'banglabert', BANGLABERT, 1.0, 'banglasarc',  'binary'),
    ('bb_fgm_e10_banglasarc3_binary',    'fgm', 'banglabert', BANGLABERT, 1.0, 'banglasarc3', 'binary'),
    ('bb_fgm_e10_banglasarc3_ternary',   'fgm', 'banglabert', BANGLABERT, 1.0, 'banglasarc3', 'ternary'),
    ('bb_fgm_e10_ben_sarc_binary',       'fgm', 'banglabert', BANGLABERT, 1.0, 'ben_sarc',    'binary'),
    # ── Group B: FreeLB ───────────────────────────────────────────────────
    ('bb_freelb_k3_banglasarc_binary',   'freelb', 'banglabert', BANGLABERT, 3, 'banglasarc',  'binary'),
    ('bb_freelb_k3_banglasarc3_binary',  'freelb', 'banglabert', BANGLABERT, 3, 'banglasarc3', 'binary'),
    ('bb_freelb_k3_banglasarc3_ternary', 'freelb', 'banglabert', BANGLABERT, 3, 'banglasarc3', 'ternary'),
    ('bb_freelb_k3_ben_sarc_binary',     'freelb', 'banglabert', BANGLABERT, 3, 'ben_sarc',    'binary'),
    # ── Group C: FGM+AWP ──────────────────────────────────────────────────
    ('bb_fgmawp_banglasarc_binary',      'fgm_awp', 'banglabert', BANGLABERT, 0.5, 'banglasarc',  'binary'),
    ('bb_fgmawp_banglasarc3_binary',     'fgm_awp', 'banglabert', BANGLABERT, 0.5, 'banglasarc3', 'binary'),
    ('bb_fgmawp_banglasarc3_ternary',    'fgm_awp', 'banglabert', BANGLABERT, 0.5, 'banglasarc3', 'ternary'),
    ('bb_fgmawp_ben_sarc_binary',        'fgm_awp', 'banglabert', BANGLABERT, 0.5, 'ben_sarc',    'binary'),
    # ── Group D: MuRIL cross-backbone ─────────────────────────────────────
    ('muril_fgm_e05_banglasarc_binary',  'fgm', 'muril', MURIL, 0.5, 'banglasarc',  'binary'),
    ('muril_fgm_e05_banglasarc3_binary', 'fgm', 'muril', MURIL, 0.5, 'banglasarc3', 'binary'),
    ('muril_fgm_e05_banglasarc3_ternary','fgm', 'muril', MURIL, 0.5, 'banglasarc3', 'ternary'),
    ('muril_fgm_e05_ben_sarc_binary',    'fgm', 'muril', MURIL, 0.5, 'ben_sarc',    'binary'),
]

print(f'Total new training runs: {len(RUN_MATRIX)}')
for r in RUN_MATRIX:
    print(f'  [{r[1]:8s}] {r[0]}')

Total new training runs: 25
  [fgm     ] bb_fgm_e03_banglasarc_binary
  [fgm     ] bb_fgm_e03_banglasarc3_binary
  [fgm     ] bb_fgm_e03_banglasarc3_ternary
  [fgm     ] bb_fgm_e03_ben_sarc_binary
  [fgm     ] bb_fgm_e05_banglasarc3_ternary
  [fgm     ] bb_fgm_e08_banglasarc_binary
  [fgm     ] bb_fgm_e08_banglasarc3_binary
  [fgm     ] bb_fgm_e08_banglasarc3_ternary
  [fgm     ] bb_fgm_e08_ben_sarc_binary
  [fgm     ] bb_fgm_e10_banglasarc_binary
  [fgm     ] bb_fgm_e10_banglasarc3_binary
  [fgm     ] bb_fgm_e10_banglasarc3_ternary
  [fgm     ] bb_fgm_e10_ben_sarc_binary
  [freelb  ] bb_freelb_k3_banglasarc_binary
  [freelb  ] bb_freelb_k3_banglasarc3_binary
  [freelb  ] bb_freelb_k3_banglasarc3_ternary
  [freelb  ] bb_freelb_k3_ben_sarc_binary
  [fgm_awp ] bb_fgmawp_banglasarc_binary
  [fgm_awp ] bb_fgmawp_banglasarc3_binary
  [fgm_awp ] bb_fgmawp_banglasarc3_ternary
  [fgm_awp ] bb_fgmawp_ben_sarc_binary
  [fgm     ] muril_fgm_e05_banglasarc_binary
  [fgm     ] muril_fgm_e05_banglas

In [39]:
# ── Cell 3: Dataset utilities ──────────────────────────────────────────────
LABEL_MAP_TERNARY = {'Non-Sarcastic': 0, 'Neutral': 1, 'Sarcastic': 2}

def load_splits(dataset_name, task):
    dfs = {}
    for split in ['train', 'val', 'test']:
        path = os.path.join(DATA_SPLITS, f'{dataset_name}_{task}_{split}.csv')
        df   = pd.read_csv(path)
        if task == 'binary':
            df['label'] = df['label_binary'].astype(int)
        else:
            df['label'] = df['label_original'].map(LABEL_MAP_TERNARY)
            df = df.dropna(subset=['label']).reset_index(drop=True)
            df['label'] = df['label'].astype(int)
        dfs[split] = df
    return dfs['train'], dfs['val'], dfs['test']


class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts   = df['text'].tolist()
        self.labels  = df['label'].tolist()
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def make_compute_metrics(num_labels):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'macro_f1':    f1_score(labels, preds, average='macro',    zero_division=0),
            'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
            'accuracy':    accuracy_score(labels, preds),
        }
    return compute_metrics


def get_word_embed_param(model):
    """Find word_embeddings weight param. Works for BERT, mBERT, MuRIL."""
    for name, param in model.named_parameters():
        if 'word_embeddings.weight' in name and param.requires_grad:
            return name, param
    return None, None


print('Dataset utilities ready.')

Dataset utilities ready.


In [40]:
# ── Cell 4: FGM + FGMTrainer ───────────────────────────────────────────────

class FGM:
    """Fast Gradient Method — single-step embedding perturbation."""
    def __init__(self, model, epsilon=0.5, emb_name='word_embeddings'):
        self.model    = model
        self.epsilon  = epsilon
        self.emb_name = emb_name
        self.backup   = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm > 1e-8:
                    param.data.add_(self.epsilon * param.grad / norm)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class _AdvTrainerBase(Trainer):
    """
    Shared base for all adversarial trainers.
    _safe_backward() is compatible with both old transformers (local, no accelerator)
    and new transformers (Kaggle, accelerator with FP16 scaler).
    """
    def _safe_backward(self, loss):
        try:
            # New transformers: accelerator handles FP16 scaling
            self.accelerator.backward(loss)
        except AttributeError:
            # Old transformers: no accelerator attribute
            if hasattr(self, 'scaler') and self.scaler is not None:
                self.scaler.scale(loss).backward()
            else:
                loss.backward()


class FGMTrainer(_AdvTrainerBase):
    def __init__(self, fgm_epsilon=0.5, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.fgm_epsilon   = fgm_epsilon
        self.class_weights = class_weights
        self.fgm           = None

    def _get_fgm(self):
        if self.fgm is None:
            unwrapped = self.model.module if hasattr(self.model, 'module') else self.model
            self.fgm  = FGM(unwrapped, epsilon=self.fgm_epsilon)
        return self.fgm

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels       = inputs.get('labels')
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs      = model(**model_inputs)
        logits       = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)
        fgm    = self._get_fgm()

        # Step 1: clean forward + backward
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        # Save BEFORE backward — graph may be freed after backward
        loss_to_return = loss.detach().clone()

        self._safe_backward(loss / self.args.gradient_accumulation_steps)

        # Step 2: adversarial forward + backward
        fgm.attack()
        with self.compute_loss_context_manager():
            loss_adv = self.compute_loss(model, inputs)
        self._safe_backward(loss_adv / self.args.gradient_accumulation_steps)
        fgm.restore()

        return loss_to_return


print('FGM + FGMTrainer ready.')

FGM + FGMTrainer ready.


In [41]:
# ── Cell 5: FreeLB + FreeLBTrainer ─────────────────────────────────────────

class FreeLBTrainer(_AdvTrainerBase):
    def __init__(
        self,
        adv_steps    = 3,
        adv_lr       = 0.04,
        adv_init_mag = 0.05,
        adv_max_norm = 0.1,
        class_weights= None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.adv_steps    = adv_steps
        self.adv_lr       = adv_lr
        self.adv_init_mag = adv_init_mag
        self.adv_max_norm = adv_max_norm
        self.class_weights= class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels       = inputs.get('labels')
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs      = model(**model_inputs)
        logits       = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        unwrapped  = model.module if hasattr(model, 'module') else model
        emb_name, embed_param = get_word_embed_param(unwrapped)

        if embed_param is None:
            with self.compute_loss_context_manager():
                loss = self.compute_loss(model, inputs)
            loss_to_return = loss.detach().clone()
            self._safe_backward(loss / self.args.gradient_accumulation_steps)
            return loss_to_return

        K          = self.adv_steps
        prev_grad  = torch.zeros_like(embed_param.data)
        delta      = torch.empty_like(embed_param.data).uniform_(
            -self.adv_init_mag, self.adv_init_mag
        )
        loss_to_return = None  # captured on first step

        for step in range(K):
            embed_param.data.add_(delta)

            with self.compute_loss_context_manager():
                loss = self.compute_loss(model, inputs)

            # Capture loss on first step before any backward
            if loss_to_return is None:
                loss_to_return = loss.detach().clone()

            self._safe_backward(loss / (K * self.args.gradient_accumulation_steps))

            embed_param.data.sub_(delta)

            if step < K - 1:
                curr_grad = embed_param.grad.detach().clone()
                step_grad = curr_grad - prev_grad
                prev_grad = curr_grad
                norm = step_grad.norm()
                if norm > 1e-8:
                    delta = delta + self.adv_lr * step_grad / norm
                delta = delta.clamp(-self.adv_max_norm, self.adv_max_norm)

        # Guaranteed non-None: K>=1 so loss_to_return is always set
        return loss_to_return


print('FreeLB + FreeLBTrainer ready.')

FreeLB + FreeLBTrainer ready.


In [42]:
# ── Cell 6: AWP + FGMAWPTrainer ────────────────────────────────────────────

class AWP:
    """Adversarial Weight Perturbation — perturbs all named 'weight' parameters."""
    def __init__(self, model, adv_param='weight', adv_lr=0.001, adv_eps=1e-3):
        self.model     = model
        self.adv_param = adv_param
        self.adv_lr    = adv_lr
        self.adv_eps   = adv_eps
        self.backup    = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if (param.requires_grad
                    and self.adv_param in name
                    and param.grad is not None):
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm > 1e-8:
                    r_at = self.adv_lr * param.grad / norm
                    param.data.add_(r_at)
                    param.data = torch.min(
                        torch.max(param.data, self.backup[name] - self.adv_eps),
                        self.backup[name] + self.adv_eps
                    )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class FGMAWPTrainer(_AdvTrainerBase):
    def __init__(self, fgm_epsilon=0.5, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.fgm_epsilon   = fgm_epsilon
        self.class_weights = class_weights
        self.fgm           = None
        self.awp           = None

    def _get_adv(self):
        if self.fgm is None or self.awp is None:
            unwrapped = self.model.module if hasattr(self.model, 'module') else self.model
            self.fgm  = FGM(unwrapped, epsilon=self.fgm_epsilon)
            self.awp  = AWP(unwrapped)
        return self.fgm, self.awp

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels       = inputs.get('labels')
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs      = model(**model_inputs)
        logits       = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs   = self._prepare_inputs(inputs)
        fgm, awp = self._get_adv()

        # Step 1: clean forward + backward
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        # Save BEFORE backward
        loss_to_return = loss.detach().clone()

        self._safe_backward(loss / self.args.gradient_accumulation_steps)

        # Step 2: AWP perturbs weights using clean gradients
        awp.attack()

        # Step 3: FGM additionally perturbs embeddings
        fgm.attack()

        # Step 4: adversarial forward + backward (both perturbations active)
        with self.compute_loss_context_manager():
            loss_adv = self.compute_loss(model, inputs)
        self._safe_backward(loss_adv / self.args.gradient_accumulation_steps)

        # Step 5: restore both
        fgm.restore()
        awp.restore()

        return loss_to_return


print('AWP + FGMAWPTrainer ready.')

AWP + FGMAWPTrainer ready.


In [43]:
# ── Cell 7: Generic training + evaluation function ─────────────────────────
import math

TRAINER_MAP = {
    'fgm':     FGMTrainer,
    'freelb':  FreeLBTrainer,
    'fgm_awp': FGMAWPTrainer,
}

def build_trainer(technique, extra_kwargs, **trainer_kwargs):
    cls = TRAINER_MAP[technique]
    return cls(**extra_kwargs, **trainer_kwargs)


def train_and_evaluate(run_id, technique, backbone_short, backbone_hf,param, dataset_name, task, completed_runs):
    if run_id in completed_runs:
        print(f'  [SKIP] {run_id}')
        return None

    sep = '=' * 72
    print(f'\n{sep}')
    print(f'  RUN  : {run_id}')
    print(f'  tech ={technique:8s}  param={param}  'f'backbone={backbone_short}  ds={dataset_name}  task={task}')
    print(sep)

    num_labels = 2 if task == 'binary' else 3
    t0 = time.time()

    # ── Data ──────────────────────────────────────────────────────────────
    train_df, val_df, test_df = load_splits(dataset_name, task)

    class_weights = None
    if task == 'ternary':
        counts        = train_df['label'].value_counts().sort_index().values.astype(float)
        w             = 1.0 / counts
        w             = w / w.sum() * num_labels
        class_weights = torch.tensor(w, dtype=torch.float32)
        print(f'  class_weights = {[round(float(x), 4) for x in w]}')

    # ── Tokenizer & datasets ───────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(backbone_hf, cache_dir=CACHE_DIR)
    train_ds  = SarcasmDataset(train_df, tokenizer)
    val_ds    = SarcasmDataset(val_df,   tokenizer)
    test_ds   = SarcasmDataset(test_df,  tokenizer)

    # ── Model ─────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        backbone_hf, num_labels=num_labels,
        ignore_mismatched_sizes=True, cache_dir=CACHE_DIR
    )
    model.gradient_checkpointing_enable()

    # ── Compute warmup_steps explicitly (warmup_ratio deprecated in v5.2) ─
    steps_per_epoch = math.ceil(len(train_ds) / (4 * 2))  # batch=4, accum=2
    total_steps     = steps_per_epoch * 3                  # 3 epochs
    warmup_steps    = max(1, int(total_steps * 0.1))

    # ── TrainingArguments ─────────────────────────────────────────────────
    run_ckpt = os.path.join(CKPT_DIR, run_id)
    os.makedirs(run_ckpt, exist_ok=True)

    training_args = TrainingArguments(
        output_dir                  = run_ckpt,
        num_train_epochs            = 3,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 16,
        gradient_accumulation_steps = 2,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        warmup_steps                = warmup_steps,   # replaces warmup_ratio
        fp16                        = USE_FP16,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        save_total_limit            = 1,
        seed                        = 42,
        report_to                   = 'none',
        logging_steps               = 50,
        dataloader_num_workers      = 0,
    )

    # ── Technique-specific kwargs ──────────────────────────────────────────
    if technique == 'fgm':
        extra = dict(fgm_epsilon=param, class_weights=class_weights)
    elif technique == 'freelb':
        extra = dict(adv_steps=int(param), class_weights=class_weights)
    elif technique == 'fgm_awp':
        extra = dict(fgm_epsilon=param, class_weights=class_weights)
    else:
        raise ValueError(f'Unknown technique: {technique}')

    trainer = build_trainer(
        technique, extra,
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = make_compute_metrics(num_labels),
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # ── Evaluation ────────────────────────────────────────────────────────
    def predict_metrics(ds):
        out   = trainer.predict(ds)
        preds = np.argmax(out.predictions, axis=-1)
        lbls  = out.label_ids
        return preds, lbls, {
            'macro_f1':    f1_score(lbls, preds, average='macro',    zero_division=0),
            'weighted_f1': f1_score(lbls, preds, average='weighted', zero_division=0),
            'accuracy':    accuracy_score(lbls, preds),
        }

    test_preds, test_lbls, test_m = predict_metrics(test_ds)
    val_preds,  val_lbls,  val_m  = predict_metrics(val_ds)
    elapsed = round(time.time() - t0, 1)

    # ── Save predictions ──────────────────────────────────────────────────
    for split, df, preds in [('test', test_df, test_preds), ('val', val_df, val_preds)]:
        out_df = df.copy(); out_df['pred'] = preds
        out_df.to_csv(os.path.join(PREDS_DIR, f'12_{run_id}_{split}_preds.csv'), index=False)

    # ── Save classification report ────────────────────────────────────────
    lnames = ['Non-Sarcastic', 'Sarcastic'] if num_labels == 2 else ['Non-Sarcastic', 'Neutral', 'Sarcastic']
    with open(os.path.join(REPORTS_DIR, f'12_{run_id}_report.txt'), 'w') as f:
        f.write(classification_report(test_lbls, test_preds, target_names=lnames, zero_division=0))

    # ── Build result record ───────────────────────────────────────────────
    result = {
        'run_id':           run_id,
        'technique':        technique,
        'backbone':         backbone_short,
        'backbone_hf':      backbone_hf,
        'param':            param,
        'dataset':          dataset_name,
        'task':             task,
        'test_macro_f1':    round(test_m['macro_f1'],    4),
        'test_weighted_f1': round(test_m['weighted_f1'], 4),
        'test_accuracy':    round(test_m['accuracy'],    4),
        'val_macro_f1':     round(val_m['macro_f1'],     4),
        'time_seconds':     elapsed,
        'num_labels':       num_labels,
    }
    with open(os.path.join(LOGS_DIR, f'12_{run_id}_log.json'), 'w') as f:
        json.dump(result, f, indent=2)

    # ── Cleanup ───────────────────────────────────────────────────────────
    shutil.rmtree(run_ckpt, ignore_errors=True)
    del model, trainer, train_ds, val_ds, test_ds, tokenizer
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    print(f'  → test_macro_f1={result["test_macro_f1"]:.4f}  elapsed={elapsed}s')
    return result


print('Training function ready.')

Training function ready.


In [44]:
# ── Cell 8: Load nb06 baseline results (FGM ε=0.5 binary) ─────────────────
# These 3 runs were completed in nb06. We pull them into nb12's results
# so the epsilon comparison table is complete without re-training.

NB06_CSV = os.path.join(TABLES_DIR, '06_fgm_results.csv')
nb06_rows = []

if os.path.exists(NB06_CSV):
    nb06_df = pd.read_csv(NB06_CSV)
    print(f'nb06 columns : {nb06_df.columns.tolist()}')
    print(nb06_df.to_string(index=False))

    for _, row in nb06_df.iterrows():
        ds = str(row.get('dataset', row.get('dataset_name', ''))).strip()
        nb06_rows.append({
            'run_id':           f'nb06_bb_fgm_e05_{ds}_binary',
            'technique':        'fgm',
            'backbone':         'banglabert',
            'backbone_hf':      BANGLABERT,
            'param':            0.5,
            'dataset':          ds,
            'task':             'binary',
            'test_macro_f1':    round(float(row['test_macro_f1']),               4),
            'test_weighted_f1': round(float(row.get('test_weighted_f1', float('nan'))), 4),
            'test_accuracy':    round(float(row.get('test_accuracy',    float('nan'))), 4),
            'val_macro_f1':     round(float(row.get('val_macro_f1',     float('nan'))), 4),
            'time_seconds':     float(row.get('time_seconds', 0)),
            'num_labels':       2,
            'source':           'nb06',
        })
    print(f'\nLoaded {len(nb06_rows)} record(s) from nb06.')
else:
    print(f'WARNING: {NB06_CSV} not found.')
    print('nb06 binary ε=0.5 data will be absent from the epsilon comparison.')
    print('Expected: 04_outputs/tables/06_fgm_results.csv')

nb06 binary ε=0.5 data will be absent from the epsilon comparison.
Expected: 04_outputs/tables/06_fgm_results.csv


In [45]:
# ── Cell 9: Main training loop (with full resume support) ──────────────────
completed_runs = set()
all_results    = []

if os.path.exists(OUTPUT_CSV):
    existing_df    = pd.read_csv(OUTPUT_CSV)
    completed_runs = set(existing_df['run_id'].tolist())
    all_results    = existing_df.to_dict('records')
    print(f'Resuming: {len(completed_runs)} run(s) already in {OUTPUT_CSV}')
else:
    print('Starting fresh.')

for (run_id, technique, backbone_short, backbone_hf,param, dataset_name, task) in RUN_MATRIX:
    result = train_and_evaluate(
        run_id, technique, backbone_short, backbone_hf,
        param, dataset_name, task, completed_runs
    )

    if result is not None:
        all_results.append(result)
        completed_runs.add(run_id)
        pd.DataFrame(all_results).to_csv(OUTPUT_CSV, index=False)
        print(f'  [Saved] {len(all_results)} run(s) → {OUTPUT_CSV}')

print(f'\nAll {len(RUN_MATRIX)} runs complete.')

Starting fresh.

  RUN  : bb_fgm_e03_banglasarc_binary
  tech =fgm       param=0.3  backbone=banglabert  ds=banglasarc  task=binary


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 50760.41it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ── Cell 10: Analysis — three comparison tables ────────────────────────────
results_df = pd.read_csv(OUTPUT_CSV)

# Merge nb06 ε=0.5 binary rows
if nb06_rows:
    nb06_aug    = pd.DataFrame(nb06_rows)
    existing_ids = set(results_df['run_id'].tolist())
    new_rows     = nb06_aug[~nb06_aug['run_id'].isin(existing_ids)]
    if not new_rows.empty:
        results_df = pd.concat([results_df, new_rows], ignore_index=True)
        print(f'Merged {len(new_rows)} nb06 row(s).')

results_df['dataset_task'] = results_df['dataset'] + '_' + results_df['task']

# ── Table 1: FGM epsilon study (BanglaBERT, Test Macro-F1) ────────────────
print('\n' + '='*72)
print('  TABLE 1 — BanglaBERT FGM: epsilon sensitivity (Test Macro-F1)')
print('='*72)

fgm_bb = results_df[
    (results_df['technique'] == 'fgm') &
    (results_df['backbone']  == 'banglabert')
].copy()

pivot_eps = fgm_bb.pivot_table(
    index='param', columns='dataset_task',
    values='test_macro_f1', aggfunc='first'
).round(4)
pivot_eps['mean'] = pivot_eps.mean(axis=1).round(4)
print(pivot_eps.sort_index().to_string())

if not fgm_bb[fgm_bb['task']=='binary'].empty:
    best_eps_binary = (
        fgm_bb[fgm_bb['task']=='binary']
        .groupby('param')['test_macro_f1'].mean()
        .sort_values(ascending=False)
    )
    print(f'\nBest ε (binary avg): ε={best_eps_binary.index[0]}  '
          f'mean={best_eps_binary.iloc[0]:.4f}')

# ── Table 2: Technique comparison at best ε (BanglaBERT, all datasets) ────
print('\n' + '='*72)
print('  TABLE 2 — Technique comparison: FGM vs FreeLB vs FGM+AWP')
print('  (BanglaBERT, Test Macro-F1)')
print('='*72)

tech_rows = []
for tech in ['fgm', 'freelb', 'fgm_awp']:
    subset = results_df[
        (results_df['technique'] == tech) &
        (results_df['backbone']  == 'banglabert')
    ]
    # For FGM use ε=0.5; for others use all rows
    if tech == 'fgm':
        subset = subset[subset['param'] == 0.5]
    if subset.empty: continue

    row = {'technique': tech}
    for dt in ['ben_sarc_binary', 'banglasarc_binary',
               'banglasarc3_binary', 'banglasarc3_ternary']:
        m = subset[subset['dataset_task'] == dt]['test_macro_f1']
        row[dt] = round(m.values[0], 4) if len(m) > 0 else None
    vals = [v for v in row.values() if isinstance(v, float)]
    row['mean'] = round(sum(vals)/len(vals), 4) if vals else None
    tech_rows.append(row)

if tech_rows:
    tech_df = pd.DataFrame(tech_rows).sort_values('mean', ascending=False)
    print(tech_df.to_string(index=False))
    best_tech = tech_df.iloc[0]['technique']
    best_mean = tech_df.iloc[0]['mean']
    print(f'\nBest technique: {best_tech}  (mean={best_mean:.4f})')

# ── Table 3: Cross-backbone — MuRIL+FGM vs BanglaBERT+FGM ─────────────────
print('\n' + '='*72)
print('  TABLE 3 — Cross-backbone: MuRIL+FGM vs BanglaBERT+FGM (ε=0.5)')
print('='*72)

muril_df  = results_df[(results_df['backbone']=='muril') & (results_df['param']==0.5)].copy()
bb_e05    = results_df[(results_df['backbone']=='banglabert') &
                       (results_df['technique']=='fgm') & (results_df['param']==0.5)].copy()

cross_rows = []
for dt in sorted(muril_df['dataset_task'].unique()):
    m_val = muril_df[muril_df['dataset_task']==dt]['test_macro_f1']
    b_val = bb_e05[bb_e05['dataset_task']==dt]['test_macro_f1']
    mv = round(m_val.values[0], 4) if len(m_val) > 0 else None
    bv = round(b_val.values[0], 4) if len(b_val) > 0 else None
    cross_rows.append({
        'dataset_task': dt,
        'muril_fgm':    mv,
        'bb_fgm':       bv,
        'delta':        round(mv-bv, 4) if (mv and bv) else None
    })

cross_df = pd.DataFrame(cross_rows)
print(cross_df.to_string(index=False))

if not muril_df.empty and not bb_e05.empty:
    print(f'\nMuRIL+FGM mean : {muril_df["test_macro_f1"].mean():.4f}')
    print(f'BB+FGM    mean : {bb_e05["test_macro_f1"].mean():.4f}')

# ── Save full results ──────────────────────────────────────────────────────
full_path = os.path.join(TABLES_DIR, '12_adv_training_full.csv')
results_df.to_csv(full_path, index=False)
print(f'\nFull results saved → {full_path}')

In [ ]:
# ── Cell 11: Grand summary ─────────────────────────────────────────────────
print('\n' + '='*72)
print('  GRAND RANKING — All adversarial techniques (mean Test Macro-F1)')
print('='*72)

summary_rows = []

# nb05 plain baseline
nb05_path = os.path.join(TABLES_DIR, '05_baseline_results.csv')
if os.path.exists(nb05_path):
    nb05 = pd.read_csv(nb05_path)
    summary_rows.append(('plain_banglabert (nb05)',
                         round(nb05['test_macro_f1'].mean(), 4)))

# nb06 FGM ε=0.5 binary
if nb06_rows:
    summary_rows.append(('bb_fgm_e05 (nb06)',
                         round(np.mean([r['test_macro_f1'] for r in nb06_rows]), 4)))

# nb12 results grouped by technique
for tech, label in [
    ('fgm',     'bb_fgm_best_eps (nb12)'),
    ('freelb',  'bb_freelb_k3 (nb12)'),
    ('fgm_awp', 'bb_fgm+awp (nb12)'),
]:
    subset = results_df[
        (results_df['technique'] == tech) &          # fixed: no more scalar True
        (results_df['backbone']  == 'banglabert')
    ]
    if subset.empty:
        continue

    if tech == 'fgm':
        # use best epsilon per dataset_task
        best_rows = subset.loc[subset.groupby('dataset_task')['test_macro_f1'].idxmax()]
        summary_rows.append((label, round(best_rows['test_macro_f1'].mean(), 4)))
    else:
        summary_rows.append((label, round(subset['test_macro_f1'].mean(), 4)))

# MuRIL cross-backbone
muril_summary = results_df[results_df['backbone'] == 'muril']
if not muril_summary.empty:
    summary_rows.append(('muril_fgm_e05 (nb12)',
                         round(muril_summary['test_macro_f1'].mean(), 4)))

if summary_rows:
    summary_df = (
        pd.DataFrame(summary_rows, columns=['method', 'mean_macro_f1'])
        .sort_values('mean_macro_f1', ascending=False)
        .reset_index(drop=True)
    )
    print(summary_df.to_string(index=False))

print('\n' + '='*72)
print('  NOTEBOOK 12 COMPLETE')
print('='*72)
print('Outputs:')
print(f'  04_outputs/tables/12_adv_training_results.csv')
print(f'  04_outputs/tables/12_adv_training_full.csv')
print(f'  04_outputs/predictions/12_*_preds.csv')
print(f'  04_outputs/reports/12_*_report.txt')
print(f'  04_outputs/run_logs/12_*_log.json')
print()
print('Next → Notebook 13: Ensemble Methods (hard/soft voting + stacking)')